# RHINO RFSoC 4x2 — CW Calibration Experiment
**University of Manchester / Jodrell Bank Observatory**  
Author: Mbatshi Jerry Junior Mbulawa  
Date: 2026-03-15

---

## What This Notebook Does

This notebook sweeps a continuous wave (CW) sine tone from **50 MHz to 200 MHz** using the RFSoC 4x2's own DAC as a signal source, and measures the ADC's response at each frequency. The result is a **gain curve** — the frequency-dependent response of the signal chain — which is the primary input to the absolute radiometer calibration method being developed for the RHINO 21cm hydrogen line experiment.

**Run the cells in order from top to bottom.** Each cell is self-contained and explains what it does.

---

## Hardware Setup (Do This Before Running)

1. Connect the **DAC_A SMA** to the **ADC_A SMA** on the RFSoC 4x2 board using an RF coax cable (loopback).
2. Make sure the board is powered on and accessible at `192.168.2.99`.
3. Copy this notebook and `rhino_cw_calibration.py` to `/home/xilinx/jupyter_notebooks/` on the board.

---

## Key Platform Facts

| Parameter | Value | Why it matters |
|---|---|---|
| DDC NCO offset | 1228.8 MHz | DAC must be at `1228.8 + f_target` MHz |
| Active ADC channel | `channel_22` | Confirmed in Session 1 (89.6 dB SNR) |
| ADC hardware sample rate | 4915.2 MSPS | Full tile rate — rfsoc_sam does **not** decimate |
| 16× decimation to 200 MSPS | Custom `system_overlay` only | **Not** a feature of rfsoc_sam |
| Noise floor | ~−107.2 dBFS | Confirmed in Session 1 |

---
## Cell 1: Imports

We import the core Python libraries and the calibration script.  
Importing `rhino_cw_calibration` gives us access to all the measurement functions without having to copy the code here.

In [ ]:
# Standard scientific Python libraries
import numpy as np                  # numerical arrays and maths
import matplotlib.pyplot as plt     # plotting
import time                         # timing and sleep

# Make plots appear inline in the notebook (not in a separate window)
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)   # default figure size
plt.rcParams['font.size'] = 11

# Import our calibration module.
# This gives us access to all the measurement functions defined in
# rhino_cw_calibration.py without duplicating the code here.
import rhino_cw_calibration as cal

print("Imports complete.")
print(f"Board connected: {cal.BOARD_CONNECTED}")
print(f"DDC offset: {cal.DDC_OFFSET_MHZ} MHz")
print(f"ADC hardware sample rate (rfsoc_sam, no decimation): {cal.ADC_FS_HZ/1e6:.1f} MSPS")
print("NOTE: 16x decimation to 200 MSPS applies to the custom system_overlay only, not rfsoc_sam.")

---
## Cell 2: Initialise Hardware

This loads the rfsoc_sam overlay onto the FPGA and gets handles to the ADC receiver (channel_22) and the DAC transmitter.  

> **If you see an error here**, check that:
> - The board is powered on and you are running this on the board itself (not a remote PC)
> - `rfsoc_sam.bit` is in `/home/xilinx/jupyter_notebooks/`

In [ ]:
# Loads the rfsoc_sam bitstream onto the FPGA.
# Also gets handles to:
#   cal._receiver    = ol.radio.receiver.channel_22   (ADC_A SMA)
#   cal._transmitter = ol.radio.transmitter.channel_00 (DAC)
# This takes a few seconds while the bitstream is programmed.

cal.initialise_hardware()
print("Hardware ready.")

---
## Cell 3: Configuration

Edit the values in this cell to change how the experiment runs.  
All other cells read from these variables — you should only need to change things here.

In [ ]:
# -----------------------------------------------------------------------
# EXPERIMENT CONFIGURATION — edit these
# -----------------------------------------------------------------------

# Frequency sweep range (MHz)
cal.SWEEP_START_MHZ = 50.0
cal.SWEEP_STOP_MHZ  = 200.0

# Step size for the sweep.
# Options:
#   1.0  MHz = fast coarse sweep (~150 steps, good for quick checks)
#   0.5  MHz = medium resolution
#   0.049 MHz = one FFT bin width, full resolution (3000+ steps, slow)
cal.SWEEP_STEP_MHZ = 1.0

# Number of spectral frames to average at each step.
# More = lower noise but slower. 20 is a good starting point.
cal.N_FRAMES = 20

# DAC amplitude (0 = off, 1 = full scale).
# Keep <= 0.7 to leave headroom and avoid ADC clipping.
cal.DAC_AMPLITUDE = 0.5

# DAC settle time after a frequency change.
# This will be automatically updated after Measurement 1.
cal.DAC_SETTLE_TIME_S = 0.1

print("Configuration set:")
print(f"  Sweep: {cal.SWEEP_START_MHZ}–{cal.SWEEP_STOP_MHZ} MHz "
      f"in {cal.SWEEP_STEP_MHZ} MHz steps")
n_steps = int((cal.SWEEP_STOP_MHZ - cal.SWEEP_START_MHZ) / cal.SWEEP_STEP_MHZ) + 1
print(f"  Number of steps: {n_steps}")
print(f"  Estimated sweep time (rough): "
      f"{n_steps * (cal.DAC_SETTLE_TIME_S + 0.1 * cal.N_FRAMES):.0f} s")

---
## Cell 4: Measurement 1 — NCO Switching Speed

**What this does:** Measures how long the DAC takes to switch from one frequency to another and stabilise.  
**Why it matters:** This sets the minimum wait time (`DAC_SETTLE_TIME_S`) between frequency steps in the sweep. Too short and you measure during a transient; too long and the sweep takes unnecessarily long.  
**What to expect:** Typically 10–100 ms. The cell will automatically update `DAC_SETTLE_TIME_S` with the correct value.

In [ ]:
# Run 10 switching trials: DAC switches from 75 MHz to 100 MHz each time.
# The function polls the ADC spectrum until the peak at 100 MHz is stable.

switch_times = cal.measure_switching_speed(n_trials=10)

if len(switch_times) > 0:
    # Update the global settle time based on the measured switching speed.
    # We use 1.5× the maximum observed switching time to be safe.
    cal.DAC_SETTLE_TIME_S = float(switch_times.max() * 1.5)
    print(f"\nSettle time updated to: {cal.DAC_SETTLE_TIME_S:.3f} s")

    # Plot the distribution of switching times
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(switch_times * 1000, bins=10, color='steelblue', edgecolor='white')
    ax.axvline(switch_times.mean() * 1000, color='red', linestyle='--',
               label=f'Mean: {switch_times.mean()*1000:.1f} ms')
    ax.axvline(switch_times.max() * 1000, color='orange', linestyle='--',
               label=f'Max: {switch_times.max()*1000:.1f} ms')
    ax.set_xlabel('Switching time (ms)')
    ax.set_ylabel('Count')
    ax.set_title('Measurement 1: NCO Switching Speed Distribution')
    ax.legend()
    plt.tight_layout()
    plt.savefig('m1_switching_speed.png', dpi=150)
    plt.show()
else:
    print("All trials timed out. Check hardware connection.")

---
## Cell 5: Measurement 4 — Noise Floor Stability

**What this does:** Records the ADC noise floor (DAC off) in the science band (60–85 MHz) over 60 seconds.  
**Why it matters:** If the noise floor drifts, calibration must be repeated frequently. This measurement tells us how thermally stable the board is.  
**What to expect:** Noise floor around −107 dBFS, stable to within ±0.5 dB over 60 seconds.

In [ ]:
# Record noise floor every 10 seconds for 60 seconds total (7 samples).
# Increase duration_s for a longer stability characterisation.

noise_mean, noise_over_time = cal.measure_noise_floor(
    duration_s=60.0,
    sample_interval_s=10.0
)

# Plot noise floor over time
t_axis = np.arange(len(noise_over_time)) * 10.0   # seconds
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_axis, noise_over_time, 'o-', color='steelblue')
ax.axhline(noise_mean, color='red', linestyle='--',
           label=f'Mean: {noise_mean:.2f} dBFS')
ax.fill_between(t_axis,
                noise_mean - 0.5, noise_mean + 0.5,
                alpha=0.2, color='green', label='±0.5 dB band')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Noise floor (dBFS)')
ax.set_title('Measurement 4: Noise Floor Stability (60–85 MHz science band)')
ax.legend()
plt.tight_layout()
plt.savefig('m4_noise_floor.png', dpi=150)
plt.show()

print(f"\nNoise floor mean: {noise_mean:.2f} dBFS")
print(f"Peak-to-peak drift: {np.ptp(noise_over_time):.2f} dB")

---
## Cell 6: Measurement 3 — Amplitude Linearity

**What this does:** Sweeps the DAC amplitude from 0.1 to 1.0 at a fixed frequency (75 MHz) and measures the ADC peak power at each level.  
**Why it matters:** For calibration to work, the system must be linear — doubling the input amplitude must produce exactly +6 dB at the ADC output. Non-linearity would mean the gain curve depends on signal level, which breaks the calibration.  
**What to expect:** A straight line with slope ≈ 1.0 on a log-log plot.

In [ ]:
# Sweep amplitude at 75 MHz (centre of the science band)
amplitudes, amp_powers = cal.measure_amplitude_linearity(
    freq_mhz=75.0,
    n_steps=10
)

# Plot: log amplitude (dB) vs measured power (dBFS)
# For a perfectly linear system, this should be a straight line with slope = 1
log_amp = 20.0 * np.log10(amplitudes)
valid = ~np.isnan(amp_powers)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(log_amp[valid], amp_powers[valid], 'o-', color='steelblue',
        label='Measured')

# Overlay the ideal linear response for comparison
if np.sum(valid) > 2:
    slope, intercept = np.polyfit(log_amp[valid], amp_powers[valid], 1)
    fit_line = slope * log_amp + intercept
    ax.plot(log_amp, fit_line, '--', color='red',
            label=f'Linear fit (slope={slope:.3f}, ideal=1.000)')

ax.set_xlabel('DAC amplitude (dB re full scale)')
ax.set_ylabel('ADC peak power (dBFS)')
ax.set_title('Measurement 3: Amplitude Linearity at 75 MHz')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('m3_linearity.png', dpi=150)
plt.show()

---
## Cell 7: Measurement 2 — Main Gain Curve Sweep

**What this does:** Sweeps the CW tone from 50 to 200 MHz and records the ADC peak power at each step. This is the primary calibration product.  
**Why it matters:** The gain curve shows how the signal chain's response varies with frequency. Science observations must be divided by this curve to get a flat, calibrated spectrum.  
**What to expect:** A smooth curve varying by roughly ±2–5 dB across the band. The science band (60–85 MHz) is highlighted.

> **Note:** This cell takes the longest to run. With `SWEEP_STEP_MHZ = 1.0` and `N_FRAMES = 20`, expect approximately 5–10 minutes.

In [ ]:
# Run the full frequency sweep.
# Uses the settle time determined in Cell 4.
# Results are automatically saved to rhino_cw_results.npz and .csv

sweep_freqs, peak_powers, peak_freqs, noise_floor = cal.run_gain_curve_sweep(
    settle_time_s=cal.DAC_SETTLE_TIME_S
)

print(f"\nSweep complete: {len(sweep_freqs)} steps measured.")
print(f"Results saved to: {cal.OUTPUT_NPZ} and {cal.OUTPUT_CSV}")

---
## Cell 8: Plot Gain Curve

Plots the full gain curve (50–200 MHz) and a zoomed view of the science band (60–85 MHz).  
The plot is also saved as a PNG file.

In [ ]:
# Plot the gain curve from the sweep results
cal.plot_gain_curve(sweep_freqs, peak_powers, noise_floor)

# Print a summary of the science band performance
sweep_arr   = np.array(sweep_freqs)
power_arr   = np.array(peak_powers)
sci_mask    = (sweep_arr >= 60) & (sweep_arr <= 85) & ~np.isnan(power_arr)

if np.any(sci_mask):
    sci_powers = power_arr[sci_mask]
    print("\nScience band summary (60–85 MHz):")
    print(f"  Mean peak power : {np.mean(sci_powers):.2f} dBFS")
    print(f"  Peak-to-peak variation: {np.ptp(sci_powers):.2f} dB")
    mean_snr = np.mean(sci_powers) - noise_floor
    print(f"  Mean SNR: {mean_snr:.1f} dB")

---
## Cell 9: Reload and Inspect Saved Results

If you want to re-examine the results without re-running the sweep, load them from the saved `.npz` file.

In [ ]:
# Load the saved results from disk
# This cell can be run independently after the sweep has been completed

data = np.load(cal.OUTPUT_NPZ)

loaded_freqs   = data['sweep_freqs']
loaded_powers  = data['peak_powers']
loaded_noise   = float(data['noise_floor'])

print(f"Loaded {len(loaded_freqs)} frequency steps from {cal.OUTPUT_NPZ}")
print(f"Frequency range: {loaded_freqs.min():.1f}–{loaded_freqs.max():.1f} MHz")
print(f"Power range: {np.nanmin(loaded_powers):.1f} to {np.nanmax(loaded_powers):.1f} dBFS")
print(f"Noise floor: {loaded_noise:.2f} dBFS")

# Quick plot of loaded data
valid = ~np.isnan(loaded_powers)
plt.figure(figsize=(12, 4))
plt.plot(loaded_freqs[valid], loaded_powers[valid], color='steelblue', lw=0.8)
plt.axhline(loaded_noise, color='grey', linestyle='--', label='Noise floor')
plt.axvspan(60, 85, alpha=0.15, color='green', label='Science band')
plt.xlabel('Frequency (MHz)')
plt.ylabel('ADC Peak Power (dBFS)')
plt.title('Loaded Gain Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary

After running all cells, you will have:

| File | Contents |
|---|---|
| `rhino_cw_results.npz` | All sweep data as numpy arrays |
| `rhino_cw_results.csv` | Same data in spreadsheet format |
| `rhino_cw_gain_curve.png` | Gain curve plot (full + science band zoom) |
| `m1_switching_speed.png` | Switching speed histogram |
| `m3_linearity.png` | Amplitude linearity plot |
| `m4_noise_floor.png` | Noise floor stability plot |

**Next steps:**
- Share the gain curve with Jordan and Phil.
- Decide whether the switching speed result requires moving to the DDS Compiler approach (Option B).
- Once the Vivado licence is resolved, repeat this experiment with the custom bitstream for comparison.